<a href="https://colab.research.google.com/github/rittikad/100daysofsql/blob/100_days_of_sql/SQLite_DB_DatSet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**STEP 1: CREATE the SQLite database;**


We need to import the sqlite3 module and create the database and tables.  You'll see this follows the syntax we have used on previous weeks.

Note that we have created the student table with a primary key that is not an INTEGER.

Is this good practice?  
What are the issues and benefits of doing this?

In [1]:
import sqlite3

#This statement creates a connection labelled as conn.  This will be used throughout to ensure the consistency for when we start to query the database tables.
conn = sqlite3.connect('ecommerce.db')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE olist_customers (
    customer_id VARCHAR(32) PRIMARY KEY,
    customer_unique_id VARCHAR(32),
    customer_zip_code_prefix INT,
    customer_city VARCHAR(255),
    customer_state VARCHAR(2)
);
''')

cursor.execute('''
CREATE TABLE olist_geolocation (
    geolocation_zip_code_prefix INT,
    geolocation_lat FLOAT,
    geolocation_lng FLOAT,
    geolocation_city VARCHAR(255),
    geolocation_state VARCHAR(2)
);
''')

cursor.execute('''
CREATE TABLE olist_order_items (
    order_id VARCHAR(32),
    order_item_id INT,
    product_id VARCHAR(32),
    seller_id VARCHAR(32),
    shipping_limit_date DATETIME,
    price FLOAT,
    freight_value FLOAT,
    PRIMARY KEY (order_id, order_item_id)
);
''')

cursor.execute('''
CREATE TABLE olist_order_payments (
    order_id VARCHAR(32),
    payment_sequential INT,
    payment_type VARCHAR(50),
    payment_installments INT,
    payment_value FLOAT,
    PRIMARY KEY (order_id, payment_sequential)
);
''')

cursor.execute('''
CREATE TABLE olist_order_reviews (
    review_id VARCHAR(32) PRIMARY KEY,
    order_id VARCHAR(32),
    review_score INT,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date DATETIME,
    review_answer_timestamp DATETIME
);
''')

cursor.execute('''
CREATE TABLE olist_orders (
    order_id VARCHAR(32) PRIMARY KEY,
    customer_id VARCHAR(32),
    order_status VARCHAR(50),
    order_purchase_timestamp DATETIME,
    order_approved_at DATETIME,
    order_delivered_carrier_date DATETIME,
    order_delivered_customer_date DATETIME,
    order_estimated_delivery_date DATETIME
);
''')

cursor.execute('''
CREATE TABLE olist_products (
    product_id VARCHAR(32) PRIMARY KEY,
    product_category_name VARCHAR(255),
    product_name_lenght FLOAT,
    product_description_lenght FLOAT,
    product_photos_qty FLOAT,
    product_weight_g FLOAT,
    product_length_cm FLOAT,
    product_height_cm FLOAT,
    product_width_cm FLOAT
);
''')

cursor.execute('''
CREATE TABLE olist_sellers (
    seller_id VARCHAR(32) PRIMARY KEY,
    seller_zip_code_prefix INT,
    seller_city VARCHAR(255),
    seller_state VARCHAR(2)
);
''')

cursor.execute('''
CREATE TABLE product_category_translation (
    product_category_name VARCHAR(255) PRIMARY KEY,
    product_category_name_english VARCHAR(255)
);
''')

#This saves the chnages to the databae.  Up unitl this point the executed SQL statement isn't stored, changes are not immediatley saved.
conn.commit()

print("Database and tables created successfully!")


Database and tables created successfully!


**STEP 2: Check Tables Created:**

Run the command to show the database tables created and the structure.

In [2]:

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

for table_name in tables:
    print(f"Table: {table_name[0]}")
    cursor.execute(f"PRAGMA table_info({table_name[0]});")
    columns = cursor.fetchall()
    for col in columns:
        print(f"  Column: {col[1]}, Type: {col[2]}, NotNull: {col[3]}, DefaultVal: {col[4]}, PrimaryKey: {col[5]}")
    print("-" * 20)




Table: olist_customers
  Column: customer_id, Type: VARCHAR(32), NotNull: 0, DefaultVal: None, PrimaryKey: 1
  Column: customer_unique_id, Type: VARCHAR(32), NotNull: 0, DefaultVal: None, PrimaryKey: 0
  Column: customer_zip_code_prefix, Type: INT, NotNull: 0, DefaultVal: None, PrimaryKey: 0
  Column: customer_city, Type: VARCHAR(255), NotNull: 0, DefaultVal: None, PrimaryKey: 0
  Column: customer_state, Type: VARCHAR(2), NotNull: 0, DefaultVal: None, PrimaryKey: 0
--------------------
Table: olist_geolocation
  Column: geolocation_zip_code_prefix, Type: INT, NotNull: 0, DefaultVal: None, PrimaryKey: 0
  Column: geolocation_lat, Type: FLOAT, NotNull: 0, DefaultVal: None, PrimaryKey: 0
  Column: geolocation_lng, Type: FLOAT, NotNull: 0, DefaultVal: None, PrimaryKey: 0
  Column: geolocation_city, Type: VARCHAR(255), NotNull: 0, DefaultVal: None, PrimaryKey: 0
  Column: geolocation_state, Type: VARCHAR(2), NotNull: 0, DefaultVal: None, PrimaryKey: 0
--------------------
Table: olist_order

**STEP 3: Upload Files:**

Run this box multiple times to upload the relevant csv files. Or drag the files across to the Files window from your desktop.

Course_Table.csv, Student_Table.csv & Grade_table.csv

In [7]:


from google.colab import files
uploaded = files.upload()
for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))


Saving olist_geolocation_dataset.csv to olist_geolocation_dataset.csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset.csv
Saving olist_products_dataset.csv to olist_products_dataset.csv
Saving .DS_Store to .DS_Store
Saving olist_customers_dataset.csv to olist_customers_dataset.csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset.csv
Saving olist_orders_dataset.csv to olist_orders_dataset.csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Saving product_category_name_translation.csv to product_category_name_translation.csv
User uploaded file "olist_geolocation_dataset.csv" with length 61273883 bytes
User uploaded file "olist_order_payments_dataset.csv" with length 5777138 bytes
User uploaded file "olist_order_reviews_dataset.csv" with length 14409007 bytes
User uploaded file "olist_products_dataset.csv" with length 2379446 bytes
User uploaded file ".DS_Store"

**STEP 4: Load CSV files into the database tables:**

This will populate the database tables with the data from teh csv files.  No need to write INSERT statements.

You need to make sure the correct files are loaded into the corresponding tables.

In [8]:
import csv

def import_csv_to_table(csv_file, table_name):
    #opens the file aas read only 'r', doesn't allow the origianl csv to be changed.
    with open(csv_file, 'r', encoding='utf-8') as file:
        csv_reader = csv.reader(file)
        next(csv_reader)  # Skip header row if present
        for row in csv_reader:
            #? creates a placeholder for each column in the CSV file. ['?','?','?'] - Join makes it a string so it can then be inserted.
            # use of the '?' reduce risk of SQL injection
            placeholders = ', '.join(['?' for _ in row])
            #Assumes that the CSV and table have the same structure (this could be an issue) Would have to specify column names if different.
            sql = f"INSERT INTO {table_name} VALUES ({placeholders})"
            cursor.execute(sql, row)

# Import data from CSV files into the relevant table - Student_Table goes into student table.  the import_csv_to_table is the function, passing the two values across.
try:
    import_csv_to_table('olist_customers_dataset.csv', 'olist_customers')
    import_csv_to_table('olist_geolocation_dataset.csv', 'olist_geolocation')
    import_csv_to_table('olist_order_items_dataset.csv', 'olist_order_items')
    import_csv_to_table('olist_order_payments_dataset.csv', 'olist_order_payments')
    #import_csv_to_table('olist_order_reviews_dataset.csv', 'olist_order_reviews')
    import_csv_to_table('olist_orders_dataset.csv', 'olist_orders')
    import_csv_to_table('olist_products_dataset.csv', 'olist_products')
    import_csv_to_table('olist_sellers_dataset.csv', 'olist_sellers')
    import_csv_to_table('product_category_name_translation.csv', 'product_category_translation')
    conn.commit()
    print("Data imported successfully!")
except Exception as e:
    print(f"An error occurred: {e}")
    conn.rollback()  # Rollback changes if an error occurred



Data imported successfully!


**STEP 5: Check Data has loaded**

Query each database table and load the data into a dataframe and display the first 5 lines

In [9]:
import pandas as pd
# Query all three tables and load into pandas DataFrames
customers_df = pd.read_sql_query("SELECT * FROM olist_customers", conn)
location_df = pd.read_sql_query("SELECT * FROM olist_geolocation", conn)
orderItems_df = pd.read_sql_query("SELECT * FROM olist_order_items", conn)

#add in the other tables


# Show the first 5 lines of each DataFrame
print("Customers Table:")
print(customers_df.head(5))
print("\nLocations Table:")
print(location_df.head(5))
print("\nOrder Itmes Table:")
print(orderItems_df.head(5))




Customers Table:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  

Locations Table:
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621   

**ONLY RUN IF YOU NEED TO DELETE THE DATA IN THE TABLES**

If you run go back to **STEP 4** and re-run from there.

In [ ]:
# only run if you need to reset the tables without deleting the databae and starting again - then re-run the box previous box.
# Delete all data from the tables
cursor.execute("PRAGMA foreign_keys = OFF")
cursor.execute("DELETE FROM olist_customers")
cursor.execute("DELETE FROM olist_geolocation")
cursor.execute("DELETE FROM olist_order_items")
cursor.execute("DELETE FROM olist_order_payments")
cursor.execute("DELETE FROM olist_order_reviews")
cursor.execute("DELETE FROM olist_orders")
cursor.execute("DELETE FROM olist_products")
cursor.execute("DELETE FROM olist_sellers")
cursor.execute("DELETE FROM product_category_translation")
cursor.execute("PRAGMA foreign_keys = ON")

# Commit the changes
conn.commit()

conn.commit()
print("Database Deleted - restart.")



**STEP 6: SQL Select statements**

Run the following statements.  Please ask yoursefl the impact of each one before running.




In [31]:
#analyse the customer list.
results = pd.read_sql_query("SELECT * FROM olist_customers", conn)
print(results)


                            customer_id                customer_unique_id  \
0      06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1      18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2      4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3      b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4      4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   
...                                 ...                               ...   
99436  17ddf5dd5d51696bb3d7c6291687be6f  1a29b476fee25c95fbafc67c5ac95cf8   
99437  e7b71a9017aa05c9a7fd292d714858e8  d52a67c98be1cf6a5c84435bd38d095d   
99438  5e28dfe12db7fb50a4b2f691faecea5e  e9f50caf99f032f0bf3c55141f019d99   
99439  56b18e2166679b8a959d72dd06da27f9  73c2643a0a458b49f58cea58833b192e   
99440  274fa6071e5e17fe303b9748641082c8  84732c5050c01db9b23e19ba39899398   

       customer_zip_code_prefix          customer_city customer_state  
0  

In [57]:
#show number and value of orders for each customer
results = pd.read_sql_query("SELECT oc.customer_id, COUNT(oop.order_id) AS number_of_orders, SUM(payment_value) AS order_value FROM olist_order_payments oop JOIN olist_orders oo ON oop.order_id = oo.order_id JOIN olist_customers oc ON oo.customer_id = oc.customer_id GROUP BY oc.customer_id", conn)
print(results)


                            customer_id  number_of_orders  order_value
0      00012a2ce6f8dcda20d059ce98491703                 1       114.74
1      000161a058600d5901f007fab4c27140                 1        67.41
2      0001fd6190edaaf884bcaf3d49edf079                 1       195.42
3      0002414f95344307404f0ace7a26f1d5                 1       179.35
4      000379cdec625522490c315e70c7a9fb                 1       107.01
...                                 ...               ...          ...
99435  fffecc9f79fd8c764f843e9951b11341                 3        81.36
99436  fffeda5b6d849fbd39689bb92087f431                 1        63.13
99437  ffff42319e9b2d713724ae527742af25                 1       214.13
99438  ffffa3172527f765de70084a7e53aae8                 1        45.50
99439  ffffe8b65bbe3087b653a978c870db99                 1        18.37

[99440 rows x 3 columns]


In [32]:
#where do the customers come from - location
results = pd.read_sql_query("SELECT customer_city, customer_state, customer_zip_code_prefix FROM olist_customers", conn)
print(results)

               customer_city customer_state  customer_zip_code_prefix
0                     franca             SP                     14409
1      sao bernardo do campo             SP                      9790
2                  sao paulo             SP                      1151
3            mogi das cruzes             SP                      8775
4                   campinas             SP                     13056
...                      ...            ...                       ...
99436              sao paulo             SP                      3937
99437        taboao da serra             SP                      6764
99438              fortaleza             CE                     60115
99439                 canoas             RS                     92120
99440                  cotia             SP                      6703

[99441 rows x 3 columns]


In [38]:
#look at which sellers are selling the most items
results = pd.read_sql_query("SELECT os.seller_id, COUNT(order_id) AS number_of_most_selling_items FROM olist_sellers os JOIN olist_order_items ooi ON ooi.seller_id = os.seller_id GROUP BY os.seller_id ORDER BY number_of_most_selling_items DESC ", conn)
print(results)


                             seller_id  number_of_most_selling_items
0     6560211a19b47992c3666cc44a7e94c0                          2033
1     4a3ca9315b744ce9f8e9374361493884                          1987
2     1f50f920176fa81dab994f9023523100                          1931
3     cc419e0650a3c5ba77189a1882b7556a                          1775
4     da8622b14eb17ae2831f4ac5b9dab84a                          1551
...                                ...                           ...
3090  04ee0ec01589969663ba5967c0e0bdc0                             1
3091  00d8b143d12632bad99c0ad66ad52825                             1
3092  00ab3eff1b5192e5f1a63bcecfee11c8                             1
3093  003554e2dce176b5555353e4f3555ac8                             1
3094  001e6ad469a905060d959994f1b41e4f                             1

[3095 rows x 2 columns]


In [73]:
#take customer that has the most orders and pull out all the individual order items - is there any consistency
results = pd.read_sql_query("SELECT customer_id, COUNT(order_id) AS number_of_orders FROM olist_orders GROUP BY 1 ORDER BY number_of_orders DESC", conn)
print(results)

                            customer_id  number_of_orders
0      ffffe8b65bbe3087b653a978c870db99                 1
1      ffffa3172527f765de70084a7e53aae8                 1
2      ffff42319e9b2d713724ae527742af25                 1
3      fffeda5b6d849fbd39689bb92087f431                 1
4      fffecc9f79fd8c764f843e9951b11341                 1
...                                 ...               ...
99436  000379cdec625522490c315e70c7a9fb                 1
99437  0002414f95344307404f0ace7a26f1d5                 1
99438  0001fd6190edaaf884bcaf3d49edf079                 1
99439  000161a058600d5901f007fab4c27140                 1
99440  00012a2ce6f8dcda20d059ce98491703                 1

[99441 rows x 2 columns]


In [69]:
# How are customer paying
results = pd.read_sql_query("SELECT DISTINCT payment_type FROM olist_order_payments", conn)
print(results)

  payment_type
0  credit_card
1       boleto
2      voucher
3   debit_card
4  not_defined


In [42]:
#Which payment method generates the most income
results = pd.read_sql_query("SELECT payment_type, SUM(payment_value) AS total_income FROM olist_order_payments GROUP BY payment_type ORDER BY total_income DESC", conn)
print(results)

  payment_type  total_income
0  credit_card  1.254208e+07
1       boleto  2.869361e+06
2      voucher  3.794369e+05
3   debit_card  2.179898e+05
4  not_defined  0.000000e+00


In [46]:
#if possible, payment methods and location - how much order value - identify the area thar spends the highest
results = pd.read_sql_query("SELECT customer_city, customer_state, payment_type, SUM(payment_value) AS total_order_value FROM olist_order_payments oop JOIN olist_orders oo ON oop.order_id = oo.order_id JOIN olist_customers oc ON oo.customer_id = oc.customer_id GROUP BY 1,2,3 ORDER BY 4 DESC", conn)
print(results)

                customer_city customer_state payment_type  total_order_value
0                   sao paulo             SP  credit_card         1757931.79
1              rio de janeiro             RJ  credit_card          942028.03
2                   sao paulo             SP       boleto          358042.07
3              belo horizonte             MG  credit_card          343387.39
4                    brasilia             DF  credit_card          291241.56
...                       ...            ...          ...                ...
7718               itaporanga             PB      voucher               1.21
7719          antonio pereira             MG      voucher               0.92
7720                 ipiranga             RS  credit_card               0.22
7721  sao sebastiao de campos             RJ  credit_card               0.19
7722                sao paulo             SP  not_defined               0.00

[7723 rows x 4 columns]


In [43]:
#What's the range of order, the min and the max
results = pd.read_sql_query("SELECT MIN(order_purchase_timestamp) AS min_order_date, MAX(order_purchase_timestamp) AS max_order_date FROM olist_orders", conn)
print(results)

        min_order_date       max_order_date
0  2016-09-04 21:15:19  2018-10-17 17:30:18


In [74]:
#What's the average order spend in the categories - show the category name in English
results = pd.read_sql("SELECT product_category_name_english, AVG(payment_value) AS average_order_value FROM olist_order_payments oop JOIN olist_order_items ooi ON oop.order_id = ooi.order_id JOIN olist_products op ON ooi.product_id = op.product_id JOIN product_category_translation pct ON op.product_category_name = pct.product_category_name GROUP BY 1 ORDER BY 2 DESC", conn)
print(results)

            product_category_name_english  average_order_value
0                               computers          1268.734318
1                         fixed_telephony           763.875498
2   small_appliances_home_oven_and_coffee           656.786154
3              agro_industry_and_commerce           471.153214
4                       home_appliances_2           464.789030
..                                    ...                  ...
66                fashion_underwear_beach            88.295417
67                                   food            88.267433
68                      cds_dvds_musicals            85.673571
69                                flowers            67.060909
70                         home_comfort_2            55.178710

[71 rows x 2 columns]
